In [ ]:
import pandas as pd
print(pd.__version__)


In [ ]:
df = pd.read_json("KAG_Appliance_energydata.json")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.info

In [ ]:
df.describe

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.dtypes


In [ ]:
df.describe()

In [ ]:
df.boxplot(figsize=(18,8))
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Lag features
df['Appliances_lag1'] = df['Appliances'].shift(1)
df['Appliances_lag2'] = df['Appliances'].shift(2)
df['Appliances_lag3'] = df['Appliances'].shift(3)

In [ ]:
df['Hour'] = df['date'].dt.hour
df['Day'] = df['date'].dt.day
df['Month'] = df['date'].dt.month
df['Weekday'] = df['date'].dt.weekday
df['Weekend'] = df['Weekday'].apply(lambda x: 1 if x >= 5 else 0)

In [ ]:
df.head()

In [ ]:
df = df.drop(columns=['date'])

In [ ]:
df.columns

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(18,12))
sns.heatmap(df.corr(), cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
corr = df.corr()['Appliances'].sort_values(ascending=False)
print(corr)

In [ ]:
df = df.drop(columns=['rv1', 'rv2'])

In [ ]:
X = df.drop('Appliances', axis=1)
y = df['Appliances']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

print("R2 Score :", r2_score(y_test, y_pred))
print("MAE :", mean_absolute_error(y_test, y_pred))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred)))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

In [ ]:
rf_pred = rf.predict(X_test)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

print("Random Forest Results")
print("----------------------")
print("R2 :", r2_score(y_test, rf_pred))
print("MAE :", mean_absolute_error(y_test, rf_pred))
print("RMSE :", np.sqrt(mean_squared_error(y_test, rf_pred)))

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

et = ExtraTreesRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

et.fit(X_train, y_train)

In [ ]:
et_pred = et.predict(X_test)

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import RandomizedSearchCV

et = ExtraTreesRegressor(random_state=42)

param_grid = {
    'n_estimators': [200,300,500,700],
    'max_depth': [10,20,30,None],
    'min_samples_split': [2,5,10],
    'min_samples_leaf': [1,2,4],
    'max_features': ['sqrt','log2',None]
}

random_search = RandomizedSearchCV(
    estimator=et,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

In [ ]:
best_model = random_search.best_estimator_

pred = best_model.predict(X_test)

print("Best Parameters")
print(random_search.best_params_)

print("R2 :", r2_score(y_test,pred))
print("MAE :", mean_absolute_error(y_test,pred))
print("RMSE :", np.sqrt(mean_squared_error(y_test,pred)))

In [ ]:
importance = pd.DataFrame({
    'Feature':X.columns,
    'Importance':best_model.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance.head(15))

In [ ]:
plt.figure(figsize=(10,7))

plt.barh(
    importance['Feature'][:15],
    importance['Importance'][:15]
)

plt.gca().invert_yaxis()

plt.title("Top 15 Important Features")

plt.show()

In [ ]:
plt.figure(figsize=(8,8))

plt.scatter(
    y_test,
    pred,
    alpha=0.5
)

plt.xlabel("Actual")

plt.ylabel("Predicted")

plt.title("Actual vs Predicted")

plt.show()

In [ ]:
import joblib

joblib.dump(best_model,"Best_Model.pkl")

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    random_state=42,
    objective="reg:squarederror"
)

xgb.fit(X_train, y_train)

In [ ]:
xgb_pred = xgb.predict(X_test)

In [ ]:
print("XGBoost Results")
print("----------------")
print("R2 :", r2_score(y_test, xgb_pred))
print("MAE :", mean_absolute_error(y_test, xgb_pred))
print("RMSE :", np.sqrt(mean_squared_error(y_test, xgb_pred)))

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Extra Trees",
        "XGBoost"
    ],
    "R2": [
        r2_score(y_test, y_pred),
        r2_score(y_test, rf_pred),
        r2_score(y_test, et_pred),
        r2_score(y_test, xgb_pred)
    ],
    "MAE": [
        mean_absolute_error(y_test, y_pred),
        mean_absolute_error(y_test, rf_pred),
        mean_absolute_error(y_test, et_pred),
        mean_absolute_error(y_test, xgb_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, y_pred)),
        np.sqrt(mean_squared_error(y_test, rf_pred)),
        np.sqrt(mean_squared_error(y_test, et_pred)),
        np.sqrt(mean_squared_error(y_test, xgb_pred))
    ]
})

results.sort_values(by="R2", ascending=False)

In [ ]:
%pip install scikit-learn

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import StackingRegressor, RandomForestRegressor
from xgboost import XGBRegressor

In [ ]:
%pip install xgboost

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

stack_model = StackingRegressor(
    estimators=[
        ('lr', LinearRegression()),
        ('rf', RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )),
        ('xgb', XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            random_state=42
        ))
    ],
    final_estimator=Ridge(),
    n_jobs=-1
)

stack_model.fit(X_train, y_train)

stack_pred = stack_model.predict(X_test)

print("Stacking Regressor")
print("R2:", r2_score(y_test, stack_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, stack_pred)))
print("MAE:", mean_absolute_error(y_test, stack_pred))

In [ ]:
# Rolling average features
df['Appliances_roll3'] = df['Appliances'].shift(1).rolling(3).mean()
df['Appliances_roll6'] = df['Appliances'].shift(1).rolling(6).mean()

In [ ]:
df = df.dropna()

In [ ]:
X = df.drop('Appliances', axis=1)
y = df['Appliances']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

et = ExtraTreesRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

et.fit(X_train, y_train)

et_pred = et.predict(X_test)

print("Extra Trees with Lag Features")
print("-----------------------------")
print("R2 :", r2_score(y_test, et_pred))
print("MAE :", mean_absolute_error(y_test, et_pred))
print("RMSE :", np.sqrt(mean_squared_error(y_test, et_pred)))

In [ ]:
import shap
import matplotlib.pyplot as plt

# Take only 100 observations for SHAP
X_shap = X_test.sample(100, random_state=42)

# Create SHAP explainer for Extra Trees
explainer = shap.TreeExplainer(et)

# Calculate SHAP values
shap_values = explainer.shap_values(
    X_shap,
    check_additivity=False
)

print("SHAP calculated successfully!")

In [ ]:
shap.summary_plot(
    shap_values,
    X_shap,
    plot_type="bar"
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test
)

In [ ]:
sample = X_test.iloc[[0]]

shap.force_plot(
    explainer.expected_value,
    explainer.shap_values(sample),
    sample,
    matplotlib=True
)

In [ ]:
df.to_csv("household_power.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt